In [3]:
def kaggle_syn_from_remote(need_update=False):
    import os
    def is_kaggle() -> bool:
        """判断当前是否在 Kaggle 云端运行"""
        return "KAGGLE_URL_BASE" in os.environ or "KAGGLE_KERNEL_RUN_TYPE" in os.environ
    if not need_update:
        return None
    if is_kaggle():
        print("kaggle_syn_from_remote()")
        print("从 march_mania 仓库同步到kaggle")
        from pathlib import Path
        from kaggle_secrets import UserSecretsClient
        secret_label = "GH_TOKEN"
        secret_value = UserSecretsClient().get_secret(secret_label)
        !git config -c user.name="MOddododo" -c user.email="wenchizhou@petalmail.com"

        # ========== 3. 安全注入 Token 到 Git 命令 ==========
        # 方法：用 Python f-string 拼接完整 shell 命令，确保变量正确替换
        repo_owner = "MOddododo"
        repo_name = "march_mania"
        branch = "dev"  # 或你的分支名
        local_dir = "/kaggle/working/code"
        token = secret_value

        # 构造带 Token 的克隆地址（Token 仅在当前命令生效，不写死到文件）
        auth_url = f"https://{token}@github.com/{repo_owner}/{repo_name}.git"


        # ========== 4. 克隆或更新仓库 ==========
        repo_path = Path(local_dir)

        if not repo_path.exists():
            print(f"📥 首次克隆: {repo_name}")
            # 使用 -b 指定分支，--depth 1 加速克隆（只拉最新提交）
            !git -c user.name="tmp" -c user.email="tmp@tmp.com" clone -b {branch} --depth 1 {auth_url} {local_dir}
        else:
            print(f"🔄 更新现有仓库: {local_dir}")
            # 使用 -C 指定工作目录，避免 cd 子 shell 问题
            !git -C {local_dir} -c user.name="tmp" -c user.email="tmp@tmp.com" pull origin {branch}

In [2]:
kaggle_syn_from_remote(False)